In [ ]:
# ===============================================================
#   Lab Assignment 6 — Hopfield Networks (Google Colab Version)
#   Includes:
#     (A) Associative Memory + Error Correction
#     (B) Eight-Rook Constraint Problem
#     (C) TSP (10 cities) using Hopfield–Tank Network
# ===============================================================

!pip install numpy matplotlib scipy --quiet

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit
import random

np.random.seed(42)
random.seed(42)

# ===============================================================
# PART A — CLASSIC HOPFIELD NETWORK (ASSOCIATIVE MEMORY)
# ===============================================================

class Hopfield:
    def __init__(self, n):
        self.n = n
        self.W = np.zeros((n, n))

    def store_patterns(self, patterns):
        """Hebbian learning with bipolar patterns {-1, +1}."""
        P = len(patterns)
        self.W = np.zeros((self.n, self.n))
        for p in patterns:
            self.W += np.outer(p, p)
        np.fill_diagonal(self.W, 0)
        self.W /= self.n

    def update_async(self, state, steps=200):
        s = state.copy()
        for _ in range(steps):
            i = np.random.randint(0, self.n)
            h = np.dot(self.W[i], s)
            s[i] = 1 if h >= 0 else -1
        return s

    def recall(self, pattern):
        return self.update_async(pattern.copy(), steps=500)


def test_error_correction(n=100, n_patterns=10, flips_list=None, trials=100):
    if flips_list is None:
        flips_list = [1,3,5,10,20,30]

    patterns = (np.random.rand(n, n_patterns) > 0.5).astype(int) * 2 - 1
    patterns = patterns.T

    hop = Hopfield(n)
    hop.store_patterns(patterns)

    print("\n===== ERROR CORRECTING CAPABILITY =====")
    for flips in flips_list:
        success = 0
        for _ in range(trials):
            idx = np.random.randint(0, n_patterns)
            pat = patterns[idx].copy()
            noisy = pat.copy()
            flip_idx = np.random.choice(n, flips, replace=False)
            noisy[flip_idx] *= -1
            rec = hop.recall(noisy)
            if np.array_equal(rec, pat):
                success += 1
        rate = success / trials
        print(f"Flips = {flips:2d} → Recall Success = {rate:.3f}")

# ===============================================================
# PART B — EIGHT ROOK PROBLEM USING HOPFIELD ENERGY
# ===============================================================

def solve_eight_rook(max_iters=15000, A=10.0):
    n = 8
    X = np.zeros((n,n), dtype=int)

    # Start with valid row-constraint (one rook per row)
    for i in range(n):
        j = np.random.randint(0, n)
        X[i,j] = 1

    def energy(X):
        E = 0
        for i in range(n):
            s = X[i].sum() - 1
            E += 0.5 * A * s * s
        for j in range(n):
            s = X[:,j].sum() - 1
            E += 0.5 * A * s * s
        return E

    Eprev = energy(X)

    for step in range(max_iters):
        i = np.random.randint(0, n)
        j = np.random.randint(0, n)
        X_try = X.copy()
        X_try[i,j] = 1 - X_try[i,j]
        Enew = energy(X_try)
        if Enew < Eprev:
            X = X_try
            Eprev = Enew

        if np.all(X.sum(axis=1)==1) and np.all(X.sum(axis=0)==1):
            print(f"Eight-rook solution found at iteration {step}")
            return X

    print("Did not converge fully, returning best found.")
    return X

# ===============================================================
# PART C — TSP USING HOPFIELD-TANK CONTINUOUS NETWORK
# ===============================================================

def random_coordinates(N=10, scale=100):
    coords = np.random.rand(N,2)*scale
    D = np.sqrt(((coords[:,None,:]-coords[None,:,:])**2).sum(axis=2))
    return D, coords

def hopfield_tsp(D, A=500, B=500, C=200, Ddamp=500,
                 dt=0.01, lam=300, max_steps=3000, tol=1e-6):
    N = D.shape[0]
    U = 0.1*np.random.randn(N,N)
    V = expit(lam*U)

    def p_next(p): return (p+1)%N

    for step in range(max_steps):
        V = expit(lam*U)
        row_sum = V.sum(axis=1)
        col_sum = V.sum(axis=0)

        net = np.zeros_like(U)

        for i in range(N):
            for p in range(N):
                termA = -A*(row_sum[i]-1)
                termB = -B*(col_sum[p]-1)
                forward = (D[i] * V[:,p_next(p)]).sum()
                backward = (D[:,i] * V[:,(p-1)%N]).sum()
                termC = -C*(forward+backward)
                termD = -Ddamp*V[i,p]
                net[i,p] = termA + termB + termC + termD

        U_prev = U.copy()
        U = U + dt * (-U + net)
        if np.linalg.norm(U-U_prev) < tol:
            print("Converged TSP dynamics at step:", step)
            break

    V = expit(lam*U)
    tour = [np.argmax(V[:,p]) for p in range(N)]

    # repair duplicates
    if len(set(tour)) != N:
        missing = [c for c in range(N) if c not in tour]
        seen=set()
        for p in range(N):
            if tour[p] in seen:
                tour[p]=missing.pop(0)
            else:
                seen.add(tour[p])

    # compute length
    length=0
    for i in range(N):
        a=tour[i]
        b=tour[(i+1)%N]
        length+=D[a,b]

    return tour, length, V

# ===============================================================
# RUN EVERYTHING (as required in your Lab Assignment)
# ===============================================================

print("========== PART A: ERROR CORRECTION ==========")
test_error_correction(n=200, n_patterns=20,
                      flips_list=[1,3,5,10,20,40,80], trials=150)

print("\n========== PART B: EIGHT ROOK ==========")
sol = solve_eight_rook()
print("Eight-rook solution matrix:\n", sol)
print("Row sums:", sol.sum(axis=1))
print("Col sums:", sol.sum(axis=0))

plt.imshow(sol, cmap='Greys')
plt.title("Eight Rook Hopfield Solution")
plt.show()

print("\n========== PART C: TSP (10 cities) ==========")
D, coords = random_coordinates(10, scale=100)
tour, length, _ = hopfield_tsp(D)
print("Proposed tour:", tour)
print("Tour length:", length)

plt.scatter(coords[:,0], coords[:,1])
for i,(x,y) in enumerate(coords):
    plt.text(x+2, y+2, str(i))
path_x=[coords[i,0] for i in tour]+[coords[tour[0],0]]
path_y=[coords[i,1] for i in tour]+[coords[tour[0],1]]
plt.plot(path_x, path_y, '-o')
plt.title(f"TSP Hopfield-Tank Tour, length={length:.2f}")
plt.show()

# Number of weights for Hopfield TSP network:
N=10
neurons=N*N
weights = neurons*(neurons-1)//2
print(f"\nFor 10-city TSP:")
print(f"Neurons = N^2 = {neurons}")
print(f"Weights needed (symmetric) = {weights}")
print("\nAll tasks completed successfully in Colab.")
